### Load Data

In [302]:
import polars as pl
import duckdb

df = pl.read_parquet("data/uc-faculty.parquet")


### Basic Data Information


In [303]:
df.describe()

statistic,year,campus,last_name,first_name,title_name,department,gross_pay,base_pay,overtime_pay,other_pay
str,f64,str,str,str,str,str,f64,f64,f64,f64
"""count""",3.660661e6,"""3660661""","""3660661""","""3660660""","""3660657""","""3660320""",3.660661e6,3.660661e6,3.660661e6,3.660661e6
"""null_count""",0.0,"""0""","""0""","""1""","""4""","""341""",0.0,0.0,0.0,0.0
"""mean""",2018.764224,null,null,null,null,null,56705.176119,48504.964208,847.00437,7353.209968
"""std""",3.458144,null,null,null,null,null,78820.283732,55530.470437,3942.074732,39005.584722
"""min""",2013.0,null,""" """,""" ""","""9964 - no description found""",null,1.0,-385300.0,-49939.0,-345255.0
"""25%""",2016.0,null,null,null,null,null,5166.0,4508.0,0.0,0.0
"""50%""",2019.0,null,null,null,null,null,34225.0,31786.0,0.0,0.0
"""75%""",2022.0,null,null,null,null,null,76860.0,71906.0,0.0,2500.0
"""max""",2024.0,null,"""zyzik ""","""ângela""","""zone mech""",null,7.061667e6,1.965054e6,298933.0,6.761667e6


In [304]:
df.head()

year,campus,last_name,first_name,title_name,department,gross_pay,base_pay,overtime_pay,other_pay
i16,cat,str,str,str,cat,i32,i32,i32,i32
2024,"""asucla""","""abbassi""","""pouria""","""dir exec""","""executive director""",390489,356085,0,34405
2024,"""asucla""","""mehdian""","""kamran""","""dir""","""systems development""",247343,236450,0,10893
2024,"""asucla""","""baker""","""donna""","""dir""","""financial planning & analysis""",243072,232372,0,10700
2024,"""asucla""","""moyer""","""michelle""","""dir""","""human resources""",241336,231051,0,10285
2024,"""asucla""","""bolton""","""cynthia""","""dir""","""rest operations""",182084,174324,0,7760


### Basic Data Filters
Table name for SQL statements is faculty_salaries

In [305]:
# Min base_pay must be higher than minimum wage full time
# To try and only have educational faculty we fuzzy match on prof, lect, instr, and adj.
duckdb.connect()
df = duckdb.sql("""
    SELECT *
    FROM df
    WHERE 
        title_name ILIKE '%prof%'
        OR title_name ILIKE '%lect%'
        OR title_name ILIKE '%adj%'
        OR title_name ILIKE '%instr%'
        -- Claude noted that we were excluding recalled faculty
        -- Do we want this in our dataset?
        OR title_name ILIKE '%recall%'
        AND base_pay >= 35152
    """).df()

duckdb.sql("""
    -- I was getting a table already exists error, so I added OR REPLACE
    CREATE OR REPLACE TABLE faculty_salaries AS
        SELECT *
        FROM df
           """)


Data info after filter

In [306]:
df.describe()

,year,gross_pay,base_pay,overtime_pay,other_pay
count,380530.000000,3.805300e+05,380530.000000,380530.000000,3.805300e+05
mean,2018.827611,1.471596e+05,100161.177156,89.234047,4.690924e+04
std,3.434375,1.540516e+05,81520.985054,1175.233186,1.017159e+05
min,2013.000000,1.000000e+00,-385300.000000,-8585.000000,-3.452550e+05
25%,2016.000000,3.010325e+04,25458.000000,0.000000,0.000000e+00
50%,2019.000000,1.128250e+05,96225.500000,0.000000,6.257000e+03
75%,2022.000000,2.089830e+05,148716.000000,0.000000,4.833300e+04
max,2024.000000,3.974061e+06,838117.000000,79214.000000,3.695451e+06


In [307]:
df.head()

,year,campus,last_name,first_name,title_name,department,gross_pay,base_pay,overtime_pay,other_pay
0,2024,asucla,tejeda,steve,electrn,maintenance facilities,79706,73479,5645,583
1,2024,berkeley,malmendier,ulrike,prof-ay-b/e/e,haas core programs,730483,548817,0,181667
2,2024,berkeley,chatman,jennifer,prof-ay-b/e/e,haas core programs,718700,445800,0,272900
3,2024,berkeley,yaghi,omar,prof-ay,dept of chemistry,690298,441667,0,248631
4,2024,berkeley,isacoff,ehud,prof-ay,molecular & cell biology,680880,481058,0,199822


## Department Cleaning

In [308]:
# Top 20 departments without cleaning 

pl.Config.set_tbl_rows(20)

duckdb.sql("""
SELECT COUNT(department) as count, department
FROM faculty_salaries
GROUP BY department
ORDER BY count DESC
           """).pl()

count,department
i64,str
5264,"""medicine"""
5135,"""intercollegiate athletics"""
5103,"""mathematics"""
3944,"""cultural & recreational affair"""
3375,"""vcsa campus recreation"""
3338,"""history"""
3252,"""law"""
3142,"""psychology"""
3028,"""economics"""


In [309]:
duckdb.sql("""
SELECT COUNT(DISTINCT department) AS unique_number_of_departments
FROM faculty_salaries
           """).pl()

unique_number_of_departments
i64
4776


## Title_name cleaning

In [ ]:
# Confirmed title_names with Claude to ensure that they were not professor roles
df = duckdb.sql("""
    SELECT *
    FROM df
    WHERE 
        -- after looking through department: intercollegiate athletics        
            -- electricians
            title_name NOT ILIKE '%electrn%'
            AND title_name NOT ILIKE '%electnr%'
            AND title_name NOT ILIKE '%electr%'
                    
            -- administrative, operational, and support roles within college athletics
            AND title_name NOT ILIKE '%profl%'
                
        -- after looking through department: m_neurology
            -- supervisors in health professions
            AND title_name NOT ILIKE '%profns%'
                
        -- after looking thorugh dentistry
            -- IT/curriculum staff role
            AND title_name NOT ILIKE '%instructional designer%'
                
        -- after looking thorugh pediatrics
            -- IT/curriculum staff role
            AND title_name NOT ILIKE '%govt%'
            AND title_name NOT ILIKE '%government%'
        
        -- Claude looked at some and then I determined if they fit
            AND title_name NOT ILIKE '%electrocardiograph%'
            AND title_name NOT ILIKE '%collections %'
            AND title_name NOT ILIKE '%intellectual property%'
            AND title_name NOT ILIKE '%adminstr%'
            AND title_name NOT ILIKE '%elected ofcr%'
            AND title_name NOT ILIKE '%instrument cntrl tchn%'
            AND title_name NOT ILIKE '%instructional design%'

    """).df()

## AFTER ALL FILTERING

In [311]:
duckdb.sql("""
    CREATE OR REPLACE TABLE faculty_salaries AS
        SELECT *
        FROM df
           """)

In [312]:
df.head()

,year,campus,last_name,first_name,title_name,department,gross_pay,base_pay,overtime_pay,other_pay
0,2024,berkeley,malmendier,ulrike,prof-ay-b/e/e,haas core programs,730483,548817,0,181667
1,2024,berkeley,chatman,jennifer,prof-ay-b/e/e,haas core programs,718700,445800,0,272900
2,2024,berkeley,yaghi,omar,prof-ay,dept of chemistry,690298,441667,0,248631
3,2024,berkeley,isacoff,ehud,prof-ay,molecular & cell biology,680880,481058,0,199822
4,2024,berkeley,farber,daniel,prof-ay-law,law,677222,478333,0,198888
